# Byte Pair Encoding Tokenizer

## Imports

In [5]:
import collections

## Encode the text into UTF-8

In [50]:
text = """
Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. 
We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). 
But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. 
I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception.
A few months ago, I got interested in Unicode and decided to spend some time learning more about it in detail. 
In this article, I’ll give an introduction to it from a programmer’s point of view.
I’m going to focus on the character set and what’s involved in working with strings and files of Unicode text. 
However, in this article I’m not going to talk about fonts, text layout/shaping/rendering, or localization in detail—those are separate issues, beyond my scope (and knowledge) here.
"""

In [51]:
tokens = text.encode('utf-8')
tokens = list(map(int, tokens))
print('Text:', text)
print('length:', len(text))
print('_____')
print('Tokens:', tokens)
print('length:', len(tokens))

Text: 
Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. 
We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). 
But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. 
I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception.
A few months ago, I got interested in Unicode and decided to spend some time learning more about it in detail. 
In this article, I’ll give an introduction to it from a programmer’s point of view.
I’m going to focus on the character set and what’s involved in working with strings and files of Unicode text. 
However, in this article I’m not going to talk about fonts, text layout/shaping/rendering, or localization in detail—those are separate issues, beyo

## Get frequent pairs

In [6]:
def get_stats(ids):
    pairs = collections.defaultdict(int)
    for i, id_ in enumerate(ids):
        if i < len(ids) - 1:
            pairs[id_, ids[i+1]] += 1
    return pairs

stats = get_stats(tokens)
#print(stats)
print(sorted(((v,k) for k,v in stats.items()), reverse=True)) #sort to see the highest number of occurences

[(32, (101, 32)), (28, (105, 110)), (21, (32, 97)), (20, (32, 116)), (18, (226, 128)), (18, (115, 32)), (17, (32, 105)), (15, (240, 159)), (15, (116, 32)), (15, (110, 32)), (15, (97, 110)), (14, (116, 104)), (13, (110, 103)), (13, (110, 100)), (13, (100, 32)), (13, (97, 114)), (12, (101, 114)), (12, (100, 101)), (11, (44, 32)), (11, (32, 115)), (10, (116, 101)), (9, (116, 105)), (9, (111, 114)), (9, (110, 116)), (9, (109, 101)), (9, (32, 102)), (8, (114, 101)), (8, (114, 32)), (8, (111, 110)), (8, (108, 101)), (8, (105, 99)), (8, (104, 101)), (8, (103, 32)), (8, (32, 111)), (7, (159, 135)), (7, (159, 133)), (7, (128, 153)), (7, (116, 111)), (7, (115, 116)), (7, (111, 117)), (7, (111, 100)), (7, (111, 32)), (7, (110, 105)), (7, (104, 97)), (7, (99, 111)), (7, (32, 119)), (6, (239, 189)), (6, (140, 240)), (6, (128, 140)), (6, (118, 101)), (6, (117, 115)), (6, (115, 44)), (6, (114, 105)), (6, (101, 115)), (6, (97, 116)), (6, (85, 110)), (6, (32, 109)), (6, (32, 100)), (6, (32, 85)), (5, (

## Merge frequent pairs

In [14]:
def merge_tokens(ids, pair, idx):
    new_ids = []
    i =0

    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1]==pair[1]:
            new_ids.append(idx)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    #print(new_ids)
    return new_ids


print(merge_tokens([5,6,7,9,1], (6,7), 99))



[5, 99, 9, 1]


In [16]:
vocab_size = 276
num_merges = vocab_size - 256
ids = list(tokens)

merges = {}
for i in range(num_merges):
    stats = get_stats(ids)
    pair = max(stats, key=stats.get)
    idx = 256 + i
    print(f'merging {pair} into a new token {idx}')
    ids = merge_tokens(ids, pair, idx)
    merges[pair] = idx

merging (101, 32) into a new token 256
merging (105, 110) into a new token 257
merging (226, 128) into a new token 258
merging (115, 32) into a new token 259
merging (240, 159) into a new token 260
merging (97, 110) into a new token 261
merging (32, 116) into a new token 262
merging (116, 32) into a new token 263
merging (97, 114) into a new token 264
merging (257, 103) into a new token 265
merging (101, 114) into a new token 266
merging (100, 32) into a new token 267
merging (44, 32) into a new token 268
merging (111, 114) into a new token 269
merging (262, 104) into a new token 270
merging (105, 99) into a new token 271
merging (111, 110) into a new token 272
merging (260, 133) into a new token 273
merging (260, 135) into a new token 274
merging (115, 116) into a new token 275


In [ ]:
vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

print(vocab)
        

In [ ]:
def decode(ids):
    tokens = b"".join(vocab[idx] for idx in ids)
    text = tokens.decode('utf-8', errors='replace')
    return text

print(decode([122]))

{0: b'\x00', 1: b'\x01', 2: b'\x02', 3: b'\x03', 4: b'\x04', 5: b'\x05', 6: b'\x06', 7: b'\x07', 8: b'\x08', 9: b'\t', 10: b'\n', 11: b'\x0b', 12: b'\x0c', 13: b'\r', 14: b'\x0e', 15: b'\x0f', 16: b'\x10', 17: b'\x11', 18: b'\x12', 19: b'\x13', 20: b'\x14', 21: b'\x15', 22: b'\x16', 23: b'\x17', 24: b'\x18', 25: b'\x19', 26: b'\x1a', 27: b'\x1b', 28: b'\x1c', 29: b'\x1d', 30: b'\x1e', 31: b'\x1f', 32: b' ', 33: b'!', 34: b'"', 35: b'#', 36: b'$', 37: b'%', 38: b'&', 39: b"'", 40: b'(', 41: b')', 42: b'*', 43: b'+', 44: b',', 45: b'-', 46: b'.', 47: b'/', 48: b'0', 49: b'1', 50: b'2', 51: b'3', 52: b'4', 53: b'5', 54: b'6', 55: b'7', 56: b'8', 57: b'9', 58: b':', 59: b';', 60: b'<', 61: b'=', 62: b'>', 63: b'?', 64: b'@', 65: b'A', 66: b'B', 67: b'C', 68: b'D', 69: b'E', 70: b'F', 71: b'G', 72: b'H', 73: b'I', 74: b'J', 75: b'K', 76: b'L', 77: b'M', 78: b'N', 79: b'O', 80: b'P', 81: b'Q', 82: b'R', 83: b'S', 84: b'T', 85: b'U', 86: b'V', 87: b'W', 88: b'X', 89: b'Y', 90: b'Z', 91: b'[',

## Encode text

In [55]:
def encode(text):
    tokens = list(text.encode('utf-8'))
    while len(tokens)  >= 2:
        stats = get_stats(tokens)
        pair = min(stats, key=lambda p: merges.get(p, float('inf')))
        if pair not in merges:
            break
        idx = merges[pair]
        tokens = merge(tokens, pair, idx)
    return tokens

print(encode('h'))

[104]


## Helper functions

In [56]:
def get_stats(ids):
    pairs = collections.defaultdict(int)
    for i, id_ in enumerate(ids):
        if i < len(ids) - 1:
            pairs[id_, ids[i+1]] += 1
    return pairs

def merge_tokens(ids, pair, idx):
    new_ids = []
    i =0

    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1]==pair[1]:
            new_ids.append(idx)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids

## Basic Tokenizer Class

Here ius how you can put everything in a simple python class

In [57]:
import collections
from collections import defaultdict

class BasicTokenizer:
    def __init__(self):
        super().__init__()

    def train(self, text, vocab_size, verbose=False):
        assert vocab_size >= 256
        self.text = text 
        self.vocab_size = vocab_size

        num_merges = vocab_size - 256
        ids = list(text.encode('utf-8'))

        merges= {}
        vocab = {idx: bytes([idx]) for idx in range(256)}


        for i in range(num_merges):
            stats = get_stats(ids)
            pair = max(stats, key=stats.get)
            #print(pair)
            idx = 256 + i
            #print(f'merging {pair} into a new token {idx}')
            ids = merge_tokens(ids,pair, idx)
            merges[pair] = idx
            #print(merges[pair])
            vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
            #print(vocab[idx])
        self.merges = merges
        self.vocab = vocab


    def encode(self,text):
        tokens = list(text.encode('utf-8')) # getting the unicode tokens from the input text
        while len(tokens)  >= 2:
            stats = get_stats(tokens)
            pair = min(stats, key=lambda p: self.merges.get(p, float('inf')))
            if pair not in self.merges:
                break
            idx = self.merges[pair]
            tokens = merge_tokens(tokens, pair, idx)
        return tokens

    def decode(self, ids):
        tokens = b"".join(self.vocab[idx] for idx in ids)
        text = tokens.decode('utf-8', errors='replace')
        return text


In [59]:
tokenizer = BasicTokenizer()

In [65]:
text = """
the cat sat on the mat
the cat ate the fat rat
the rat sat on a mat
a cat and a rat and a bat
"""

In [66]:
tokenizer.train(text, vocab_size=266)
tokenizer.merges

{(97, 116): 256,
 (256, 32): 257,
 (101, 32): 258,
 (116, 104): 259,
 (259, 258): 260,
 (256, 10): 261,
 (97, 32): 262,
 (99, 257): 263,
 (32, 262): 264,
 (260, 263): 265}

In [67]:
ids = tokenizer.encode("the bat sat on the cat and the rat ate a mat")
ids

[260,
 98,
 257,
 115,
 257,
 111,
 110,
 32,
 265,
 97,
 110,
 100,
 32,
 260,
 114,
 257,
 256,
 258,
 262,
 109,
 256]

In [75]:
for id in ids:
    print(id, '-->', tokenizer.decode([id]))

260 --> the 
98 --> b
257 --> at 
115 --> s
257 --> at 
111 --> o
110 --> n
32 -->  
265 --> the cat 
97 --> a
110 --> n
100 --> d
32 -->  
260 --> the 
114 --> r
257 --> at 
256 --> at
258 --> e 
262 --> a 
109 --> m
256 --> at
